# 03 Feature Engineering
This notebook transforms clean race data into **predictive features** —
the mathematical language that machine learning models use to make predictions.

**Before feature engineering**, the model only sees raw isolated numbers:
- `position: 3, points: 15, grid: 2`

**After feature engineering**, it sees meaningful context:
- `driver_avg_finish_last_3_races: 4.33`  → Is this driver in good form right now?
- `driver_avg_finish_on_circuit: 2.8`     → Does this driver excel at this circuit?
- `constructor_reliability: 0.91`          → Is this car likely to finish?
- `driver_beats_teammate_rate: 0.72`       → Is this driver faster than their teammate?

# This notebook creates **19 new features** in 4 groups:

| Group                      | Features | What It Captures                              |
|----------------------------|----------|-----------------------------------------------|
| **1. Driver Form**         | 7        | Recent momentum, season standing, reliability |
| **2. Constructor Form**    | 5        | Team performance, reliability, championship   |
| **3. Track Performance**   | 5        | Circuit-specific strengths and weaknesses     |
| **4. Teammate Comparison** | 2        | Driver skill isolated from car performance    |


# Why these Four Groups?

**Driver form** captures momentum. A driver on a 3-race podium streak is in a
different state than one scoring zero points in 3 races.

**Constructor form** captures team dominance. In F1, the car accounts for ~70–80%
of lap time. Red Bull 2022–2023 or Mercedes 2014–2020 dominance is a real signal.

**Track performance** captures circuit specialization. Monaco requires a completely
different driving style than Monza. Historical track records are highly predictive.

**Teammate comparison** isolates driver talent from car advantage. Both teammates
race identical machinery — any performance gap reflects pure driver skill.


# Import Libs

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.3f}'.format)

print('Libs imported successfully')

# Load Data

In [ ]:
df = pd.read_csv('../data/processed/f1_clean_2018_2026.csv')

original_col_count = df.shape[1]

In [ ]:
print(f'shape: {df.shape[0]:,} rows x {df.shape[1]} columns')
print(f' Seasons: {df.season.min()} - {df.season.max()}')
print(f' Unique drivers: {df.driver_id.nunique()}')
print(f' Unique constructors: {df.constructor_id.nunique()}')
print(f' Unique circuit: {df.circuit_id.nunique()}')

print('Available columns:')
print(df.columns.tolist())


# Prevent Data leakage

In [ ]:
# sort chronologocaly

df = df.sort_values(['season', 'round']).reset_index(drop=True)

#temp int col to simplify .sum



#conver dnf/podium from bool to int

df['dnf'] = df['dnf'].astype(int)
df['podium'] = df['podium'].astype(int)

In [ ]:
df['win'] = (df['position'] == 1).astype(int)

In [ ]:

print(f'  First entry: Season {df.season.iloc[0]}, Round {df.rounds.iloc[0]}')
print(f'  Last entry:  Season {df.season.iloc[-1]}, Round {df.rounds.iloc[-1]}')
print(f'  Helper columns added: win, DNF (int), podium (int)')

# 1 Driver Form

In [ ]:
df['driver_avg_finish_last_3_races'] = (
    df.groupby('driver_id')['position']
    .transform(lambda x: x.shift(1).rolling(window=3, min_periods=1).mean())
)

In [ ]:
df['driver_avg_points_last_3_races'] = (
    df.groupby('driver_id')['position']
    .transform(lambda x: x.shift(1).rolling(window=3, min_periods=1).mean())
)

In [ ]:
df['driver_avg_grid_last_3_races'] =(
    df.groupby('driver_id')['position']
    .transform(lambda x: x.shift(1).rolling(window=3, min_periods=1).mean())
)

# Check Rolling

In [ ]:
sample_driver = df.sort_values('driver_id').driver_id.iloc[0]

verify_df = df[df.driver_id == sample_driver][[
    'season', 'round', 'position', 'points', 'grid',
    'driver_avg_finish_last_3_races',
    'driver_avg_points_last_3_races',
    'driver_avg_grid_last_3_races'
]]

In [ ]:
verify_df.head(8)

In [ ]:
verify_df.to_string(index=False)

# Season Cumulative Stats

In [ ]:
df['driver_points_cur_season'] = (
    df.groupby(['driver_id', 'season'])['points']
    .transform(lambda x: x.shift(1).expanding().sum())
    .fillna(0)
)

In [ ]:
df['driver_wins_cur_season'] = (
    df.groupby(['driver_id', 'season'])['win']
    .transform(lambda x: x.shift(1).expanding().sum())
    .fillna(0)
)

In [ ]:
df['driver_podiums_cur_season'] = (
    df.groupby(['driver_id', 'season'])['podium']
    .transform(lambda x: x.shift(1).expanding().sum())
    .fillna(0)
)   


# DNF Rate

In [ ]:
df['driver_dnf_rate'] = (
  df.groupby('driver_id')['dnf']
  .transform(lambda x: x.shift(1).expanding().mean())
 )

In [ ]:
section1_features = [
    'driver_avg_finish_last_3_races',
    'driver_avg_points_last_3_races',
    'driver_avg_grid_last_3_races',
    'driver_points_cur_season',
    'driver_wins_cur_season',
    'driver_podiums_cur_season',
    'driver_dnf_rate'
]

In [ ]:
print('=' * 65)
print(f'SECTION 1 COMPLETE: {len(section1_features)} driver form features')
print('=' * 65)
print(f'{"Feature":<44} {"NaN":>8} {"NaN %":>8}')
print('-' * 65)
for col in section1_features:
    n   = df[col].isna().sum()
    pct = df[col].isna().mean() * 100
    print(f'{col:<44} {n:>8,} {pct:>7.1f}%')
print()
print('NaN values are EXPECTED — they appear for the first few races')
print('of each driver career. Will be filled in Section 5.')

# 2 Constructor Form

In [ ]:
constructor_race = (
    df.groupby(['season', 'round', 'constructor_id'])
    .agg(
        constructor_avg_finish_race = ('position', 'mean'),
        constructor_points_race = ('points', 'sum'),
        constructor_dnf_race = ('dnf', 'mean'),
        constructor_win_flag = ('win', 'max')
    )
    .reset_index()
    .sort_values(['season', 'round'])
    .reset_index(drop=True)
)

In [ ]:
print(f'Constructor race summary shape: {constructor_race.shape}')
print(f'Expected rows: {df.groupby(["season", "round", "constructor_id"]).ngroup}')
constructor_race.head(6)

# Constructor Rolling Featurs

In [ ]:
constructor_race['constructor_avg_finish_last_3_races'] = (
    constructor_race.groupby('constructor_id')['constructor_avg_finish_race']
    .transform(lambda x: x.shift(1).rolling(3, min_periods=1).mean())
)

In [ ]:
constructor_race['constructor_avg_points_last_3_races'] =(
    constructor_race.groupby('constructor_id')['constructor_points_race']
    .transform(lambda x: x.shift(1).rolling(3, min_periods=1).mean())
)

In [ ]:
constructor_race['constructor_points_current_season'] = (
    constructor_race.groupby(['constructor_id', 'season'])['constructor_points_race']
    .transform(lambda x: x.shift(1).expanding().sum())
    .fillna(0)
)

In [ ]:
constructor_race['constructor_wins_current_season'] = (
    constructor_race.groupby(['constructor_id', 'season'])['constructor_win_flag']
    .transform(lambda x: x.shift(1).expanding().sum())
    .fillna(0)
)

In [ ]:
constructor_race['constructor_reliability'] =(
    1 - constructor_race.groupby('constructor_id')['constructor_dnf_race']
    .transform(lambda x: x.shift(1).expanding().mean())
)

In [ ]:
constructor_cols_to_merge = [
    'season', 'round', 'constructor_id',
    'constructor_avg_finish_last_3_races',
    'constructor_avg_points_last_3_races',
    'constructor_points_current_season',
    'constructor_wins_current_season',
    'constructor_reliability'
]

In [ ]:
df = df.merge(
    constructor_race[constructor_cols_to_merge],
    on=['season', 'round', 'constructor_id'],
    how='left'
)

In [ ]:
section2_features = [
    'constructor_avg_finish_last_3_races',
    'constructor_avg_points_last_3_races',
    'constructor_points_current_season',
    'constructor_wins_current_season',
    'constructor_reliability'
]

In [ ]:
print(f'Constructor features merged. Dataset shape: {df.shape}')
print()
print('=' * 65)
print(f'SECTION 2 COMPLETE: {len(section2_features)} constructor features')
print('=' * 65)
print(f'{"Feature":<48} {"NaN":>8}')
print('-' * 65)
for col in section2_features:
    n = df[col].isna().sum()
    print(f'{col:<48} {n:>8,}')

# 3 Track Performance

In [ ]:
df['driver_avg_finish_on_circuit'] = (
    df.groupby(['driver_id', 'circuit_id'])['position']
    .transform(lambda x: x.shift(1).expanding().mean())
)

In [ ]:
df['driver_avg_points_on_circuit'] = (
    df.groupby(['driver_id', 'circuit_id'])['points']
    .transform(lambda x: x.shift(1).expanding().mean())
)

In [ ]:
df['driver_best_finish_on_circuit'] = (
    df.groupby(['driver_id', 'circuit_id'])['position']
    .transform(lambda x: x.shift(1).expanding().min())
)

In [ ]:
df['driver_dnf_on_circuit'] = (
    df.groupby(['driver_id', 'circuit_id'])['dnf']
    .transform(lambda x: x.shift(1).expanding().mean())
)

# Track Strength Score

In [ ]:
df['driver_track_strength_score'] = 21.0 - df['driver_avg_finish_on_circuit']

section3_features = [
    'driver_avg_finish_on_circuit',
    'driver_avg_points_on_circuit',
    'driver_best_finish_on_circuit',
    'driver_dnf_on_circuit',
    'driver_track_strength_score'
]

In [ ]:
print('  ✅ driver_track_strength_score')
print()
print('=' * 65)
print(f'SECTION 3 COMPLETE: {len(section3_features)} track performance features')
print('=' * 65)
print(f'{"Feature":<44} {"NaN":>8} {"NaN %":>8}')
print('-' * 65)
for col in section3_features:
    n   = df[col].isna().sum()
    pct = df[col].isna().mean() * 100
    print(f'{col:<44} {n:>8,} {pct:>7.1f}%')
print()
print('Note: NaN values are expected here — first visit to each circuit.')
print('They will be filled with neutral defaults in Section 5.')

# 4 Teammate Comparison

In [ ]:
base_cols = ['season', 'round', 'constructor_id', 'driver_id', 'position']


In [ ]:
teammate_df = (
    df[base_cols].merge(
        df[base_cols].rename(columns={
            'driver_id': 'teammate_id',
            'position': 'teammate_position'
        }),
        on=['season', 'round', 'constructor_id']
    )
)

In [ ]:
teammate_df = teammate_df.sort_values(
    ['season', 'round', 'driver_id', 'teammate_id']
)

In [ ]:
teammate_df = teammate_df.drop_duplicates(
    subset=['season', 'round', 'driver_id'],
    keep='first'
)

# per-race comparison

In [ ]:
#1 finish ahead
teammate_df['beat_teammate'] = (
    teammate_df.position < teammate_df.teammate_position
).astype(int)

In [ ]:
teammate_df['gap_vs_teammate'] =(
    teammate_df.position - teammate_df.teammate_position
)

In [ ]:
teammate_df = teammate_df.sort_values(['season', 'round']).reset_index(drop=True)

print(f'Teammate matchup table created: {teammate_df.shape}')
print()

In [ ]:
print(teammate_df[[
    'season', 'round', 'driver_id', 'teammate_id',
    'position', 'teammate_position', 'beat_teammate', 'gap_vs_teammate'
]].head(6).to_string(index=False))

# Historical Teammate Rates

In [ ]:
teammate_df['driver_beats_teammate_rate'] = (
    teammate_df.groupby('driver_id')['beat_teammate']
    .transform(lambda x: x.shift(1).expanding().mean())
)

In [ ]:
teammate_df['driver_vs_teammate_finish_gap'] =(
    teammate_df.groupby('driver_id')['gap_vs_teammate']
    .transform(lambda x: x.shift(1).expanding().mean())
)

# Merge to main df

In [ ]:
df = df.merge(
    teammate_df[[
        'season', 'round', 'driver_id',
        'driver_beats_teammate_rate',
        'driver_vs_teammate_finish_gap'
    ]],
    on=['season', 'round', 'driver_id'],
    how='left'
)

In [ ]:
section4_features = [
    'driver_beats_teammate_rate',
    'driver_vs_teammate_finish_gap'
]

In [ ]:
print(f'Teammate features merged. Dataset shape: {df.shape}\n')
print('='*65)
print(f'SECTION 4 COMPLETE: {len(section4_features)} teammate features')
print('='*65)
for col in section4_features:
    n = df[col].isna().sum()
    pct = df[col].isna().mean()*100
    print(f'{col}: {n:,} NaN ({pct:1f})')

# NaN analysis & filling

In [ ]:
all_new_features = (
    section1_features
    + section2_features
    + section3_features
    + section4_features
)

In [ ]:
print('=' * 70)
print('NaN ANALYSIS — All 19 Engineered Features')
print('=' * 70)
print(f'{"Feature":<46} {"NaN Count":>12} {"NaN %":>8}')
print('-' * 70)

for col in all_new_features:
    n   = df[col].isna().sum()
    pct = df[col].isna().mean() * 100
    print(f'{col:<46} {n:>12,} {pct:>7.1f}%')

print('-' * 70)
total = df[all_new_features].isna().sum().sum()
print(f'{"TOTAL":<46} {total:>12,}')
print()
nan_free = (~df[all_new_features].isna().any(axis=1)).sum()
print(f'Rows with zero NaN in features: {nan_free:,} / {len(df):,}')

# Fill NaN

In [ ]:
global_avg_finish = df.position.mean()
global_avg_grid = df.grid.mean()

print(f'Global average finishing position: {global_avg_finish}')
print(f'Global average grid position: {global_avg_grid}')


In [ ]:
# ─── DRIVER FORM FEATURES ──────────────────────────────────
# Unknown recent form → assume average/neutral performance
df['driver_avg_finish_last_3_races'].fillna(global_avg_finish, inplace=True)
df['driver_avg_points_last_3_races'].fillna(0,                 inplace=True)
df['driver_avg_grid_last_3_races'].fillna(global_avg_grid,     inplace=True)
df['driver_dnf_rate'].fillna(0.10,                             inplace=True)
# Season cumulative stats already filled with 0 at creation time ✅

# ─── CONSTRUCTOR FEATURES ──────────────────────────────────
df['constructor_avg_finish_last_3_races'].fillna(global_avg_finish, inplace=True)
df['constructor_avg_points_last_3_races'].fillna(0,                 inplace=True)
df['constructor_reliability'].fillna(0.85,                          inplace=True)
# Season cumulative stats already filled with 0 at creation time ✅

# ─── TRACK PERFORMANCE FEATURES ────────────────────────────
# No circuit history → assume driver performs at global average
df['driver_avg_finish_on_circuit'].fillna(global_avg_finish,       inplace=True)
df['driver_avg_points_on_circuit'].fillna(0,                       inplace=True)
df['driver_best_finish_on_circuit'].fillna(global_avg_finish,      inplace=True)
df['driver_dnf_on_circuit'].fillna(0.10,                           inplace=True)
df['driver_track_strength_score'].fillna(21.0 - global_avg_finish, inplace=True)

# ─── TEAMMATE COMPARISON FEATURES ──────────────────────────
# No teammate history → assume perfectly neutral performance
df['driver_beats_teammate_rate'].fillna(0.5,    inplace=True)
df['driver_vs_teammate_finish_gap'].fillna(0.0, inplace=True)

# ─── VERIFY ────────────────────────────────────────────────
remaining = df[all_new_features].isna().sum().sum()
print('NaN filling complete!')
print(f'Remaining NaN in engineered features: {remaining}')
assert remaining == 0, 'ERROR: NaN values still remain!'
print('All 19 features are fully populated. Ready for ML.')

In [ ]:
df.columns

# Overview

In [ ]:
df.drop(columns=['win'], inplace=True)

print('='*70)
print(f'Total rows: {df.shape[0]:,}')
print(f'Total columns: {df.shape[1]}')
print(f'Original columns: {original_col_count}')
print(f'New features: {len(all_new_features)}')

In [ ]:
groups = {
    'Driver Form(7)': section1_features,
    'Constructor Form(5)': section2_features,
    'Track Performance(5)': section3_features,
    'Teammate Comparison(2)': section4_features,
}

In [ ]:
for group_name, features in groups.items():
    print(f'{group_name}:')
    for f in features:
        print(f'{f}')
    print()

# Save

In [ ]:
output_path = '../data/processed/f1_features_2018_2026.csv'
df.to_csv(output_path, index=False)

In [ ]:
df_verify = pd.read_csv(output_path)
ok = df_verify.shape == df.shape

In [ ]:
print(f'File: {output_path}')
print(f'Shape: {df.shape[0]:,} rows x {df.shape[1]} columns')
print(f'Verification: {"Pass" if ok else "Fail - shape mismatch"}')